## Import code, download model and install dependencies

In [ ]:
import subprocess

try:
    subprocess.run(['nvidia-smi'], check=True)
except Exception as exc:
    raise RuntimeError(
        'GPU not detected. In Colab, open Runtime → Change runtime type, select GPU (or TPU if GPUs are unavailable), save, then rerun this cell.'
    ) from exc


In [ ]:
from pathlib import Path

WAN2GP_ROOT = Path('/content/ltx-server').resolve()
print(f'ltx  will be installed to: {WAN2GP_ROOT}')


In [ ]:
import subprocess

repo_url = 'https://github.com/hoangth-protonx/ltx-server.git'
if WAN2GP_ROOT.exists():
    print('Repository already exists. Pulling latest changes...')
    subprocess.run(['git', '-C', str(WAN2GP_ROOT), 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', repo_url, str(WAN2GP_ROOT)], check=True)


In [ ]:
%cd ltx-server

# !git checkout feaute/serving-ltx

In [ ]:
import os, subprocess

env = os.environ.copy()
env['DEBIAN_FRONTEND'] = 'noninteractive'

subprocess.run(['sudo', 'apt-get', 'update', '-qq'], check=True, env=env)
subprocess.run([
    'sudo', 'apt-get', 'install', '-y', '--no-install-recommends',
    'ffmpeg', 'libglib2.0-0', 'libgl1', 'libportaudio2'
], check=True, env=env)


In [ ]:
import os, subprocess, sys

env = os.environ.copy()
env.setdefault('DEBIAN_FRONTEND', 'noninteractive')

subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'], check=True, env=env)
# subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch==2.8.0', 'torchvision', 'torchaudio', '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True, env=env)
subprocess.run([sys.executable, '-m', 'pip', 'install',
    '--force-reinstall', '--no-deps',
    'torch==2.8.0', 'torchvision==0.23.0', 'torchaudio==2.8.0',
    '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True, env=env)

subprocess.run([sys.executable, "-m", "pip", "install", "xformers==0.0.32.post2", "--index-url", "https://download.pytorch.org/whl/cu128"],
               check=True, env=env)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(WAN2GP_ROOT / 'requirements.txt')], check=True, env=env)

In [ ]:
from huggingface_hub import snapshot_download, login

login(token="")

REPO_ID = "DeepBeepMeep/LTX-2"

allow_patterns = [
    # "ltx-2.3-22b-dev_diffusion_model.safetensors",
    "ltx-2.3-22b-distilled_diffusion_model.safetensors", # replace with ltx-2.3-22b-distilled_diffusion_model_quanto_int8.safetensors for T4 GPU
    "ltx-2.3-22b_vae.safetensors",
    "ltx-2.3-22b_audio_vae.safetensors",
    "ltx-2.3-22b_vocoder.safetensors",
    "ltx-2.3-22b_text_embedding_projection.safetensors",
    "ltx-2.3-22b_embeddings_connector.safetensors",
    "ltx-2.3-spatial-upscaler-x2-1.1.safetensors",
    "gemma-3-12b-it-qat-q4_0-unquantized/**",
]

snapshot_download(
    repo_id=REPO_ID,
    repo_type="model",
    local_dir="ckpts",
    local_dir_use_symlinks=False,
    allow_patterns=allow_patterns,
)

print("Done. Files downloaded into ./ckpts")

In [ ]:
# !python run_ltx2.py \
#   --prompt "A spaceship landing on Mars" \
#   --gemma_path /content/wan2gp-fixed-torch-cmp/ckpts/gemma-3-12b-it-qat-q4_0-unquantized \
#   --transformer_path /content/wan2gp-fixed-torch-cmp/ckpts/ltx-2.3-22b-distilled_diffusion_model.safetensors \
#   --profile 1 \
#   --dtype bf16 \
#   --width 768 \
#   --height 512 \
#   --attention flash \
#   --num_frames 121 \
#   --num_inference_steps 30 \
#   --guidance_scale 4.0 \
#   --vram_safety_coefficient 0.75 \
#   --output_dir output

In [ ]:
!pip install -r requirements.txt

In [ ]:
!pip install nest_asyncio pyngrok

## Restart session and run the server

In [ ]:
from pyngrok import ngrok

# Set your auth token (get from ngrok dashboard)
ngrok.set_auth_token("")

In [ ]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
public_url = ngrok.connect(8000)
print(public_url)

In [ ]:
%cd ltx-server

In [ ]:
from pathlib import Path

config_path = Path("/content/ltx-server/src/config.py")

new_code = r'''
"""Server configuration management"""

import os
from pathlib import Path
from dataclasses import dataclass, field


@dataclass
class ServerConfig:
    """Server configuration"""
    # Model configuration
    model_type: str = "ltx2_22B"
    lora_dir: str = "/content/ltx-server/ckpts/loras"
    transformer_path: str = "/content/ltx-server/ckpts/ltx-2.3-22b-distilled_diffusion_model.safetensors"  # Path to transformer safetensors file(s)
    gemma_path: str = "/content/ltx-server/ckpts/gemma-3-12b-it-qat-q4_0-unquantized/gemma-3-12b-it-qat-q4_0-unquantized_quanto_bf16_int8.safetensors"  # Path to Gemma text encoder safetensors file
    profile: int = 1
    vram_safety_coefficient: float = 0.1

    # Server configuration
    output_dir: str = "output"
    host: str = "0.0.0.0"
    port: int = 8000
    reload: bool = False
    upload_dir: str = "uploads"

    def __post_init__(self):
        """Create directories if they don't exist"""
        Path(self.output_dir).mkdir(parents=True, exist_ok=True)
        Path(self.upload_dir).mkdir(parents=True, exist_ok=True)

    @property
    def num_inference_steps_default(self) -> int:
        """Default inference steps based on model type"""
        return 40 if "19B" in self.model_type else 30

'''

config_path.write_text(new_code, encoding="utf-8")
print(f"Saved: {config_path}")

In [ ]:
import uvicorn

from src.main import create_app

app = create_app()

uvicorn.run(app, host="0.0.0.0", port=8000)

## Add lora adapter

In [ ]:
# @title
from huggingface_hub import snapshot_download, login

login(token="")

REPO_ID = "AviadDahan/LTX-2.3-ID-LoRA-TalkVid-3K"

allow_patterns = [
    "lora_weights.safetensors",
]

snapshot_download(
    repo_id=REPO_ID,
    repo_type="model",
    local_dir="/content/ltx-server/ckpts/loras",
    local_dir_use_symlinks=False,
    allow_patterns=allow_patterns,
)

print("Done. Files downloaded into ./ckpts")

In [ ]:
# @title
from huggingface_hub import snapshot_download, login

login(token="")

REPO_ID = "DeepBeepMeep/LTX-2"

allow_patterns = [
    "ltx-2.3-22b-ic-lora-union-control-ref0.5.safetensors",

]

snapshot_download(
    repo_id=REPO_ID,
    repo_type="model",
    local_dir="/content/ltx-server/ckpts/loras",
    local_dir_use_symlinks=False,
    allow_patterns=allow_patterns,
)

print("Done. Files downloaded into ./ckpts")